## Arreglo Base de Datos

In [51]:
import pandas as pd
import re

# 1. Cargar la base de datos
df = pd.read_csv('madrid_ciudad_accidentes.csv', low_memory=False)

### Filtrar por conductores

In [52]:
# 5. Filtrar solo conductores
df['tipo_persona_aux'] = df['tipo_persona'].astype(str).str.strip().str.lower()
df = df[df['tipo_persona_aux'] == 'conductor'].copy()

### Arreglo columnas minúscula/mayúscula

In [53]:
# Diccionario para unificar el histórico de Madrid
mapeo_accidentes = {
    # 0: Atropellos
    'atropello': 0, 'atropello a persona': 0, 'atropello a animal': 0,
    
    # 1: Colisiones (Genéricas)
    'colisión doble': 1, 'colisión múltiple': 1, 'alcance': 1, 
    'colisión frontal': 1, 'colisión fronto-lateral': 1, 'colisión lateral': 1,
    
    # 2: Choques contra objetos
    'choque con objeto fijo': 2, 'choque contra obstáculo fijo': 2,
    
    # 3: Caídas
    'caída motocicleta': 3, 'caída bicicleta': 3, 'caída ciclomotor': 3, 
    'caída vehículo 3 ruedas': 3, 'caída viajero bus': 3, 'caída': 3,
    
    # 4: Otros / Vuelcos
    'vuelco': 4, 'solo salida de la vía': 4, 'despeñamiento': 4,
    'otras causas': 5, 'otro': 5
}

In [54]:
# 2. Limpieza básica: Todo a minúsculas y quitar espacios
columnas_texto = ['distrito', 'tipo_accidente', 'tipo_vehiculo', 'sexo', 'rango_edad', 'estado_meteorológico']
for col in columnas_texto:
    df[col] = df[col].astype(str).str.strip().str.lower()

# 3. Unificar Distritos (Quitar tildes y nombres antiguos)
reemplazos_distritos = {
    'chamartin': 'chamartín', 'chamberi': 'chamberí', 'tetuan': 'tetuán',
    'vicalvaro': 'vicálvaro', 'san blas': 'san blas-canillejas'
}
df['distrito'] = df['distrito'].replace(reemplazos_distritos)

# 4. Aplicar el Diccionario de Accidentes (del paso anterior)
df['id_tipo_accidente'] = df['tipo_accidente'].map(mapeo_accidentes)

,fecha,hora,dia_semana,distrito,localizacion,numero,num_expediente,CPFA Granizo,CPFA Hielo,CPFA Lluvia,...,rango_edad,cod_distrito,estado_meteorológico,cod_lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga,tipo_persona_aux,id_tipo_accidente
8,2016-01-01,DE 00:00 A 00:59,VIERNES,puente de vallecas,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...,0,2016/4,NO,NO,NO,...,de 30 a 34 anos,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,0.0
9,2016-01-01,DE 1:00 A 1:59,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,NO,NO,NO,...,de 30 a 34 anos,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,1.0
10,2016-01-01,DE 1:00 A 1:59,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,NO,NO,NO,...,de 25 a 29 años,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,1.0
11,2016-01-01,DE 2:00 A 2:59,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,NO,NO,NO,...,de 50 a 54 años,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,1.0
12,2016-01-01,DE 2:00 A 2:59,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,NO,NO,NO,...,de 50 a 54 años,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311021,2023-12-31,21:15:00,NaN,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,NaN,NaN,...,de 45 a 49 años,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,conductor,1.0
311022,2023-12-31,21:15:00,NaN,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,NaN,NaN,...,de 60 a 64 años,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,conductor,1.0
311025,2023-12-29,09:35:00,NaN,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,NaN,NaN,...,de 45 a 49 años,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,conductor,1.0
311026,2023-12-29,09:35:00,NaN,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,NaN,NaN,...,de 21 a 24 años,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,conductor,1.0


### Diccionario Coordenadas

In [55]:
import re

# 2. Estandarizar Direcciones para el cruce
# Creamos una columna 'direccion_unica' combinando localizacion y numero
df['direccion_unica'] = df['localizacion'].astype(str).str.strip().str.upper() + " " + df['numero'].astype(str).str.strip()

# 3. Crear diccionario de coordenadas de referencia (usando datos de 2020+)
# Esto sirve para rellenar los huecos de 2016-2019
referencia_coordenadas = df.dropna(subset=['coordenada_x_utm', 'coordenada_y_utm'])
diccionario_x = referencia_coordenadas.set_index('direccion_unica')['coordenada_x_utm'].to_dict()
diccionario_y = referencia_coordenadas.set_index('direccion_unica')['coordenada_y_utm'].to_dict()

# 4. Rellenar coordenadas vacías en 2016-2019 usando el diccionario
df['coordenada_x_utm'] = df['coordenada_x_utm'].fillna(df['direccion_unica'].map(diccionario_x))
df['coordenada_y_utm'] = df['coordenada_y_utm'].fillna(df['direccion_unica'].map(diccionario_y))

,fecha,hora,dia_semana,distrito,localizacion,numero,num_expediente,CPFA Granizo,CPFA Hielo,CPFA Lluvia,...,cod_distrito,estado_meteorológico,cod_lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga,tipo_persona_aux,id_tipo_accidente,direccion_unica
8,2016-01-01,DE 00:00 A 00:59,VIERNES,puente de vallecas,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...,0,2016/4,NO,NO,NO,...,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,0.0,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...
9,2016-01-01,DE 1:00 A 1:59,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,NO,NO,NO,...,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
10,2016-01-01,DE 1:00 A 1:59,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,NO,NO,NO,...,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
11,2016-01-01,DE 2:00 A 2:59,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,NO,NO,NO,...,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
12,2016-01-01,DE 2:00 A 2:59,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,NO,NO,NO,...,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311021,2023-12-31,21:15:00,NaN,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,NaN,NaN,...,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,conductor,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
311022,2023-12-31,21:15:00,NaN,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,NaN,NaN,...,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,conductor,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
311025,2023-12-29,09:35:00,NaN,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,NaN,NaN,...,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,conductor,1.0,"PTA. TOLEDO, 0 0"
311026,2023-12-29,09:35:00,NaN,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,NaN,NaN,...,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,conductor,1.0,"PTA. TOLEDO, 0 0"


### Limpiar hora

In [56]:
#Limpiar Hora (de rangos a hora fija)
def limpiar_hora(texto):
    match = re.search(r'(\d{1,2}:\d{2})', str(texto))
    if match:
        h = match.group(1)
        if len(h.split(':')[0]) == 1: h = '0' + h
        return f"{h}:00"
    return "08:00:00" # Por defecto si falla, o puedes poner lo que prefieras

df['hora'] = df['hora'].apply(limpiar_hora)

,fecha,hora,dia_semana,distrito,localizacion,numero,num_expediente,CPFA Granizo,CPFA Hielo,CPFA Lluvia,...,cod_distrito,estado_meteorológico,cod_lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga,tipo_persona_aux,id_tipo_accidente,direccion_unica
8,2016-01-01,00:00:00,VIERNES,puente de vallecas,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...,0,2016/4,NO,NO,NO,...,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,0.0,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...
9,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,NO,NO,NO,...,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
10,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,NO,NO,NO,...,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
11,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,NO,NO,NO,...,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
12,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,NO,NO,NO,...,NaN,nan,NaN,NaN,NaN,NaN,NaN,conductor,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311021,2023-12-31,21:15:00,NaN,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,NaN,NaN,...,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,conductor,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
311022,2023-12-31,21:15:00,NaN,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,NaN,NaN,...,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,conductor,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
311025,2023-12-29,09:35:00,NaN,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,NaN,NaN,...,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,conductor,1.0,"PTA. TOLEDO, 0 0"
311026,2023-12-29,09:35:00,NaN,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,NaN,NaN,...,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,conductor,1.0,"PTA. TOLEDO, 0 0"


### Estandarizar Clima y eliminar columnas CPFA/CPSV

In [57]:
def mapear_clima(row):
    # 1. Si ya tiene un valor en 'estado_meteorológico' (datos 2019+), lo mantenemos
    if pd.notna(row['estado_meteorológico']) and str(row['estado_meteorológico']).lower() != 'nan':
        return str(row['estado_meteorológico']).lower().strip()
    
    # 2. Mapeo de columnas antiguas (2016-2018) a las nuevas categorías
    # El orden importa: priorizamos fenómenos más graves
    if row.get('CPFA Granizo') == 'SI': return 'granizando'
    if row.get('CPFA Nieve') == 'SI':   return 'nevando'
    if row.get('CPFA Lluvia') == 'SI':  return 'lluvia débil'
    if row.get('CPFA Niebla') == 'SI':  return 'niebla'
    if row.get('CPFA Seco') == 'SI':    return 'despejado'
    if row.get('CPFA Hielo') == 'SI':   return 'helando' # El hielo suele ir a 'se desconoce' o 'otros'
    
    return 'se desconoce'

# Aplicamos la función
df['estado_meteorológico'] = df.apply(mapear_clima, axis=1)

# Listado de todas las columnas a eliminar (clima y estado del pavimento)
cols_drop = [
    'CPFA Granizo', 'CPFA Hielo', 'CPFA Lluvia', 'CPFA Niebla', 'CPFA Seco', 'CPFA Nieve',
    'CPSV Mojada', 'CPSV Aceite', 'CPSV Barro', 'CPSV Grava Suelta', 'CPSV Hielo', 'CPSV Seca Y Limpia',
    'tipo_persona_aux'
]

# Eliminamos solo las que existan en el DataFrame para evitar errores
df.drop(columns=[c for c in cols_drop if c in df.columns], inplace=True)

,fecha,hora,dia_semana,distrito,localizacion,numero,num_expediente,num_victimas,tipo_accidente,tipo_vehiculo,...,rango_edad,cod_distrito,estado_meteorológico,cod_lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga,id_tipo_accidente,direccion_unica
8,2016-01-01,00:00:00,VIERNES,puente de vallecas,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...,0,2016/4,4.0,atropello,turismo,...,de 30 a 34 anos,NaN,despejado,NaN,NaN,NaN,NaN,NaN,0.0,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...
9,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,1.0,colisión doble,turismo,...,de 30 a 34 anos,NaN,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
10,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,1.0,colisión doble,motocicleta,...,de 25 a 29 años,NaN,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
11,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,1.0,colisión múltiple,turismo,...,de 50 a 54 años,NaN,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
12,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,1.0,colisión múltiple,turismo,...,de 50 a 54 años,NaN,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311021,2023-12-31,21:15:00,NaN,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,colisión fronto-lateral,turismo,...,de 45 a 49 años,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
311022,2023-12-31,21:15:00,NaN,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,colisión fronto-lateral,turismo,...,de 60 a 64 años,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
311025,2023-12-29,09:35:00,NaN,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,alcance,motocicleta hasta 125cc,...,de 45 a 49 años,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,1.0,"PTA. TOLEDO, 0 0"
311026,2023-12-29,09:35:00,NaN,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,alcance,turismo,...,de 21 a 24 años,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,1.0,"PTA. TOLEDO, 0 0"


### Día de la semana

In [58]:
# 1. Asegúrate de convertir la columna a datetime primero
# El parámetro errors='coerce' evitará que el código falle si hay alguna fecha mal escrita
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')

# 2. Ahora ya puedes usar .dt sin que de error
df['dia_semana_ing'] = df['fecha'].dt.day_name()

# 3. Traducir a español (opcional)
dias = {
    'Monday': 'LUNES', 'Tuesday': 'MARTES', 'Wednesday': 'MIERCOLES',
    'Thursday': 'JUEVES', 'Friday': 'VIERNES', 'Saturday': 'SABADO', 'Sunday': 'DOMINGO'
}
df['dia_semana'] = df['dia_semana_ing'].map(dias)

# (Opcional) Borrar la columna en inglés si no la necesitas
df.drop(columns=['dia_semana_ing'], inplace=True)

,fecha,hora,dia_semana,distrito,localizacion,numero,num_expediente,num_victimas,tipo_accidente,tipo_vehiculo,...,rango_edad,cod_distrito,estado_meteorológico,cod_lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga,id_tipo_accidente,direccion_unica
8,2016-01-01,00:00:00,VIERNES,puente de vallecas,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...,0,2016/4,4.0,atropello,turismo,...,de 30 a 34 anos,NaN,despejado,NaN,NaN,NaN,NaN,NaN,0.0,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...
9,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,1.0,colisión doble,turismo,...,de 30 a 34 anos,NaN,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
10,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,1.0,colisión doble,motocicleta,...,de 25 a 29 años,NaN,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
11,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,1.0,colisión múltiple,turismo,...,de 50 a 54 años,NaN,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
12,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,1.0,colisión múltiple,turismo,...,de 50 a 54 años,NaN,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311021,2023-12-31,21:15:00,DOMINGO,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,colisión fronto-lateral,turismo,...,de 45 a 49 años,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
311022,2023-12-31,21:15:00,DOMINGO,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,colisión fronto-lateral,turismo,...,de 60 a 64 años,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
311025,2023-12-29,09:35:00,VIERNES,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,alcance,motocicleta hasta 125cc,...,de 45 a 49 años,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,1.0,"PTA. TOLEDO, 0 0"
311026,2023-12-29,09:35:00,VIERNES,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,alcance,turismo,...,de 21 a 24 años,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,1.0,"PTA. TOLEDO, 0 0"


### Código distrito

In [59]:
import pandas as pd

# 2. Limpieza previa de la columna 'distrito' 
# Es vital porque en 2016-2018 hay espacios extras y mayúsculas (ej. "RETIRO          ")
df['distrito_clean'] = df['distrito'].astype(str).str.strip().str.upper()

# 3. Crear el diccionario de referencia
# Filtramos las filas donde 'cod_distrito' NO es nulo y creamos el mapa
df_con_codigo = df.dropna(subset=['cod_distrito'])
mapa_distritos = pd.Series(
    df_con_codigo.cod_distrito.values, 
    index=df_con_codigo.distrito_clean
).to_dict()

# 4. Rellenar los huecos en la columna original
# Si 'cod_distrito' es NaN, buscamos el nombre del distrito en nuestro mapa
df['cod_distrito'] = df['cod_distrito'].fillna(df['distrito_clean'].map(mapa_distritos))

# 5. Verificación y limpieza
nulos_despues = df['cod_distrito'].isna().sum()
print(f"Códigos de distrito recuperados correctamente.")
print(f"Filas que aún no tienen código (probablemente distritos desconocidos o NaN): {nulos_despues}")

# Eliminamos la columna auxiliar y guardamos
df.drop(columns=['distrito_clean'], inplace=True)

Códigos de distrito recuperados correctamente.
Filas que aún no tienen código (probablemente distritos desconocidos o NaN): 7


,fecha,hora,dia_semana,distrito,localizacion,numero,num_expediente,num_victimas,tipo_accidente,tipo_vehiculo,...,rango_edad,cod_distrito,estado_meteorológico,cod_lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga,id_tipo_accidente,direccion_unica
8,2016-01-01,00:00:00,VIERNES,puente de vallecas,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...,0,2016/4,4.0,atropello,turismo,...,de 30 a 34 anos,13.0,despejado,NaN,NaN,NaN,NaN,NaN,0.0,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...
9,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,1.0,colisión doble,turismo,...,de 30 a 34 anos,15.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
10,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,1.0,colisión doble,motocicleta,...,de 25 a 29 años,15.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
11,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,1.0,colisión múltiple,turismo,...,de 50 a 54 años,5.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
12,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,1.0,colisión múltiple,turismo,...,de 50 a 54 años,5.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311021,2023-12-31,21:15:00,DOMINGO,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,colisión fronto-lateral,turismo,...,de 45 a 49 años,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
311022,2023-12-31,21:15:00,DOMINGO,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,colisión fronto-lateral,turismo,...,de 60 a 64 años,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
311025,2023-12-29,09:35:00,VIERNES,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,alcance,motocicleta hasta 125cc,...,de 45 a 49 años,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,1.0,"PTA. TOLEDO, 0 0"
311026,2023-12-29,09:35:00,VIERNES,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,alcance,turismo,...,de 21 a 24 años,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,1.0,"PTA. TOLEDO, 0 0"


### Guardar Resultado Final

In [60]:
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
df_final = df[df['fecha'].dt.year >= 2016].copy()

df_final

,fecha,hora,dia_semana,distrito,localizacion,numero,num_expediente,num_victimas,tipo_accidente,tipo_vehiculo,...,rango_edad,cod_distrito,estado_meteorológico,cod_lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga,id_tipo_accidente,direccion_unica
8,2016-01-01,00:00:00,VIERNES,puente de vallecas,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...,0,2016/4,4.0,atropello,turismo,...,de 30 a 34 anos,13.0,despejado,NaN,NaN,NaN,NaN,NaN,0.0,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...
9,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,1.0,colisión doble,turismo,...,de 30 a 34 anos,15.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
10,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,1.0,colisión doble,motocicleta,...,de 25 a 29 años,15.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
11,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,1.0,colisión múltiple,turismo,...,de 50 a 54 años,5.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
12,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,1.0,colisión múltiple,turismo,...,de 50 a 54 años,5.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311021,2023-12-31,21:15:00,DOMINGO,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,colisión fronto-lateral,turismo,...,de 45 a 49 años,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
311022,2023-12-31,21:15:00,DOMINGO,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,colisión fronto-lateral,turismo,...,de 60 a 64 años,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
311025,2023-12-29,09:35:00,VIERNES,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,alcance,motocicleta hasta 125cc,...,de 45 a 49 años,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,1.0,"PTA. TOLEDO, 0 0"
311026,2023-12-29,09:35:00,VIERNES,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,alcance,turismo,...,de 21 a 24 años,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,1.0,"PTA. TOLEDO, 0 0"


In [62]:
df_final.to_csv('madrid_accidentes_limpio.csv', index=False)

print("Proceso completo: Coordenadas recuperadas, horas fijas y clima estandarizado.")

Proceso completo: Coordenadas recuperadas, horas fijas y clima estandarizado.


## Coordenadas

In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# 1. Supongamos que ya tienes tu df con algunos nulos tras el diccionario
# Filtramos solo las filas que siguen siendo nulas para no gastar recursos
df_nulos = df_final[df_final['coordenada_x_utm'].isna()].copy()

# 2. Creamos la dirección completa para el buscador (importante añadir "Madrid, España")
df_nulos['direccion_completa'] = df_nulos['localizacion'].astype(str) + ", " + df_nulos['numero'].astype(str) + ", Madrid, España"

# 3. Configurar el geocodificador (Nominatim es gratuito)
geolocator = Nominatim(user_agent="recuperador_accidentes_madrid")
# Limitamos a 1 petición por segundo (norma de uso de Nominatim)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

print(f"Buscando coordenadas para {len(df_nulos)} direcciones únicas...")

# 4. Obtener la ubicación (esto puede tardar si son muchos registros)
# Se recomienda hacerlo por direcciones únicas primero para ir más rápido
direcciones_unicas = df_nulos['direccion_completa'].unique()
mapa_geo = {}

for dir in direcciones_unicas:
    try:
        location = geolocator.geocode(dir)
        if location:
            # Guardamos latitud y longitud
            mapa_geo[dir] = (location.latitude, location.longitude)
    except:
        continue

# 5. Aplicar los resultados al DataFrame de nulos
df_nulos['coords'] = df_nulos['direccion_completa'].map(mapa_geo)

# 6. Separar Latitud y Longitud
df_nulos['latitud'] = df_nulos['coords'].apply(lambda x: x[0] if x else None)
df_nulos['longitud'] = df_nulos['coords'].apply(lambda x: x[1] if x else None)

print(df_nulos[['direccion_completa', 'latitud', 'longitud']].head())

Buscando coordenadas para 56967 direcciones únicas...


In [ ]:
df_final['dir_ref'] = df_todo['localizacion'].astype(str).str.strip() + " " + df_todo['numero'].astype(str).str.strip()

# Nos quedamos solo con los que tienen coordenadas para hacer el diccionario
ref = df_todo.dropna(subset=['coordenada_x_utm', 'coordenada_y_utm']).drop_duplicates('dir_ref')
dict_x = pd.Series(ref.coordenada_x_utm.values, index=ref.dir_ref).to_dict()
dict_y = pd.Series(ref.coordenada_y_utm.values, index=ref.dir_ref).to_dict()

# 2. Ahora aplicamos este diccionario a tus datos de 2016-2019
df_filtrado['dir_buscar'] = df_filtrado['localizacion'].astype(str).str.strip() + " " + df_filtrado['numero'].astype(str).str.strip()

# AQUÍ ES DONDE SE OBTIENEN LAS COORDENADAS
df_filtrado['coordenada_x_utm'] = df_filtrado['dir_buscar'].map(dict_x)
df_filtrado['coordenada_y_utm'] = df_filtrado['dir_buscar'].map(dict_y)

# 3. Guardar el resultado con las coordenadas ya rellenas
df_filtrado.to_csv('coordenadas_recuperadas_2016_2019.csv', index=False)

## Reporte Automático

In [61]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df_final, title="Análisis Exploratorio Final Accidentes Madrid")
profile.to_file("ArregloFinal.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 38.71it/s]
